# 08. Topic Pool Revision Test

승인된 review decision을 기반으로 topic pool revision proposal을 생성합니다.

In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import taxonomy.topic_pool_revision as topic_pool_revision

importlib.reload(config_loader)
importlib.reload(topic_pool_revision)

from common.config_loader import load_config, get_output_table
from taxonomy.topic_pool_revision import build_and_save_topic_pool_revisions

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")
print("config loaded")

In [ ]:
MODEL_KEY = "gpt_55"
MODEL_VERSION_VALUE = config["llm"]["models"][MODEL_KEY]["model_version"]
TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-01. 채널/컨텐츠 APP"
TARGET_SC_MEASUREMENT = 1

review_decision_table = get_output_table(config, "review_decision")

display(
    spark.table(review_decision_table)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == config["version"]["prompt_version"])
    .where(F.col("taxonomy_version") == config["version"]["taxonomy_version"])
    .groupBy("decision_status", "approved_action")
    .agg(F.count("*").alias("decision_cnt"))
    .orderBy("decision_status", "approved_action")
)

In [ ]:
result = build_and_save_topic_pool_revisions(
    spark=spark,
    config=config,
    cate_1_depth=TARGET_CATE_1_DEPTH,
    cate_2_depth=TARGET_CATE_2_DEPTH,
    sc_measurement=TARGET_SC_MEASUREMENT,
    model_version=MODEL_VERSION_VALUE,
    prompt_version=config["version"]["prompt_version"],
    taxonomy_version=config["version"]["taxonomy_version"],
    write_mode="replace_revisions",
)

result

In [ ]:
topic_pool_revision_table = get_output_table(config, "topic_pool_revision")

display(
    spark.table(topic_pool_revision_table)
    .where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
    .where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
    .where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == config["version"]["prompt_version"])
    .where(F.col("taxonomy_version") == config["version"]["taxonomy_version"])
    .orderBy(F.col("created_at").desc(), F.col("revision_type").asc())
)